In [2]:
from warnings import filterwarnings
filterwarnings("ignore")

# Upgrade Pillow first to avoid PIL._typing version conflicts
!pip install --upgrade Pillow
!pip install --upgrade pdfplumber sentence-transformers scikit-learn openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 74.5 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os._exit(0)

In [1]:
import re
import numpy as np
import pandas as pd
import pdfplumber

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
from openai import OpenAI
import json
client = OpenAI(api_key="")

#Job data & embedding

In [16]:
jobs_df = pd.read_csv("/content/final_jobs_cleaned.csv")

In [17]:
job_embeddings = np.load("job_embeddings.npy")

#Extract text from resume



In [18]:
def extract_information(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

#Clean resume

In [19]:
def clean_resume_text(text):

    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )

    text = re.sub(
        r'(\+?\d[\d\s\-\(\)]{7,})',
        ' ',
        text
    )

    text = re.sub(
        r'https?://\S+',
        ' ',
        text
    )

    text = re.sub(
        r'\b(linkedin|github)\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()

#Build job context for GPT

In [20]:
def build_job_context(row):

    return f"""

Job Title:

{row['title']}

Skills:

{row['Skills']}

Experience:

{row['experience']}

Education:

{row['education']}

Job Description:

{row['job_text']}

"""

# GPT skill gaps analysis

In [28]:
def gpt_skill_gap_analysis(
    resume_text,
    job_context,
    match_score
):

    prompt = f"""
You are an expert technical recruiter.

Analyze the candidate resume against the job requirements.

Resume:
{resume_text}

Job:
{job_context}

Match Score:
{match_score:.2%}

Instructions:

1. Use only information explicitly present.
2. Consider project experience as evidence of skills.
3. Consider equivalent technologies and synonyms.
4. Do not invent qualifications.
5. Focus on technical and professional requirements.

Return valid JSON only:

{{
    "matched_skills": [],
    "missing_skills": [],
    "skill_gap_summary": "",
    "learning_recommendations": [],
    "recommendation_explanation": ""
}}
"""

    response = client.responses.create(
        model="gpt-5.4",
        input=prompt
    )

    return response.output_text

Embedding Model & resume Embeddings

In [29]:
model = SentenceTransformer("all-MiniLM-L6-v2")

def build_cv_embedding(cv_text):
    return model.encode(cv_text)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Cosine similarity

In [30]:
def compute_final_score(cv_emb, job_emb):

    semantic_score = cosine_similarity([cv_emb], [job_emb])[0][0]

    return semantic_score

# RETRIEVE TOP JOBS

In [31]:
def recommend_jobs(
    pdf_path,
    top_k=10
):

    resume_text = extract_information(
        pdf_path
    )

    resume_text = clean_resume_text(
        resume_text
    )

    cv_emb = build_cv_embedding(
        resume_text
    )

    scores = []

    for job_emb in job_embeddings:

        score = compute_final_score(
            cv_emb,
            job_emb
        )

        scores.append(score)

    temp = jobs_df.copy()

    temp["score"] = scores

    temp = temp.sort_values(
        "score",
        ascending=False
    )

    temp = temp.drop_duplicates(
        subset=["title", "name"],
        keep="first"
    )

    return temp.head(top_k)


# GPT ANALYSIS FOR TOP 5 JOBS

In [32]:
def analyze_top_jobs(
    top_df,
    resume_text
):

    results = []

    for _, row in top_df.head(5).iterrows():

        job_context = build_job_context(
            row
        )

        analysis = gpt_skill_gap_analysis(
            resume_text=resume_text,
            job_context=job_context,
            match_score=row["score"]
        )

        results.append(
            {
                "title": row["title"],
                "company": row["name"],
                "score": row["score"],
                "analysis": analysis
            }
        )

    return results

# Result

In [33]:
def print_results(results):

    for i, result in enumerate(
        results,
        start=1
    ):

        print("\n" + "=" * 80)

        print(
            f"{i}. {result['title']}"
        )

        print(
            f"Company: {result['company']}"
        )

        print(
            f"Match Score: {result['score']:.2%}"
        )

        try:

            analysis = json.loads(
                result["analysis"]
            )

            print("\nMatched Skills:")

            for skill in analysis[
                "matched_skills"
            ]:

                print(
                    f"✓ {skill}"
                )

            print(
                "\nMissing Skills:"
            )

            for skill in analysis[
                "missing_skills"
            ]:

                print(
                    f"• {skill}"
                )

            print(
                "\nSkill Gap Summary:"
            )

            print(
                analysis[
                    "skill_gap_summary"
                ]
            )

            print(
                "\nLearning Recommendations:"
            )

            for item in analysis[
                "learning_recommendations"
            ]:

                print(
                    f"- {item}"
                )

            print(
                "\nExplanation:"
            )

            print(
                analysis[
                    "recommendation_explanation"
                ]
            )

        except:

            print(
                result["analysis"]
            )
# put in the resume
cv_path = (
    "/content/Noura_Alhuthali_Resume.pdf"
)

resume_text = extract_information(
    cv_path
)

resume_text = clean_resume_text(
    resume_text
)

top_jobs = recommend_jobs(
    cv_path,
    top_k=10
)

results = analyze_top_jobs(
    top_jobs,
    resume_text
)

print_results(
    results
)


1. data science / data analysis
Company: saudia airlines
Match Score: 70.12%

Matched Skills:
✓ Data analysis
✓ Analysis
✓ Critical thinking
✓ Problem-solving
✓ Data visualization
✓ Machine learning
✓ Statistics
✓ Research
✓ Insight extraction
✓ Data preprocessing
✓ Structured and unstructured data handling
✓ Predictive modeling
✓ SQL
✓ Python
✓ Power BI
✓ Tableau
✓ EDA
✓ Data cleaning
✓ Statistical analysis
✓ NLP
✓ Deep learning
✓ LLMs

Missing Skills:
• Automating data collection processes
• Ensemble modeling
• Collaboration with engineering and product development teams
• Explicit aviation domain experience
• Explicit math-focused evidence
• STEP English test score of 72 or above
• Official GPA evidence meeting minimum threshold

Skill Gap Summary:
The candidate shows strong alignment with the technical core of the role through academic background in Data Science, hands-on ML/NLP projects, SQL-based analysis, predictive modeling, preprocessing, and dashboarding with Power BI. Resum

# Evaluation

In [61]:
from sklearn.metrics.pairwise import cosine_similarity

def build_evaluation_dataset(pdf_path, cv_name, top_k=10):

    resume_text = clean_resume_text(pdf_path)

    cv_embedding = model.encode(resume_text)

    scores = cosine_similarity(
        [cv_embedding],
        job_embeddings
    )[0]

    temp_df = jobs_df.copy()
    temp_df["score"] = scores

    temp_df = temp_df.sort_values("score", ascending=False)
    temp_df = temp_df.drop_duplicates(subset=["title", "name"], keep="first")

    top_jobs = temp_df.head(top_k).copy()

    top_jobs["cv_name"] = cv_name
    top_jobs["cv_text"] = resume_text

    return top_jobs[[
        "cv_name",
        "cv_text",
        "title",
        "name",
        "job_text",
        "score"
    ]]

In [62]:
all_results = []

cv_files = [
    ("/content/Noura_Alhuthali_Resume.pdf", "Data Science"),
    ("/content/Data analysis .pdf", "Data Analyst"),
    ("/content/SoftwareEngineerResume.pdf", "Software Engineer"),
    ("/content/4. Cyber Security Specialist - Resume.pdf", "Cybersecurity")
]

for path, name in cv_files:
    all_results.append(recommend_jobs(path, name))

evaluation_df = pd.concat(all_results, ignore_index=True)

In [63]:
print(len(evaluation_df))

40


In [64]:
evaluation_df.to_csv(
    "evaluation_pairs.csv",
    index=False
)

In [65]:
evaluation_df = pd.read_csv(
    "evaluation_pairs.csv"
)

In [66]:
PROMPT = """
You are an expert recruitment evaluator.

Return ONLY ONE INTEGER number.

Valid values:
0, 1, 2, 3

Do not write anything else.
No words. No explanation. No punctuation.

Candidate Profile:
{cv_text}

Job Description:
{job_text}
"""

In [67]:
from openai import OpenAI

client = OpenAI(api_key="")

def judge_relevance(cv_text, job_text):

    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=PROMPT.format(
                cv_text=cv_text[:3000],
                job_text=job_text[:3000]
            )
        )

        # extract first digit only
        for ch in response.output_text:
            if ch.isdigit():
                return int(ch)

        return 0

    except:
        return 0

In [68]:
evaluation_df["relevance"] = evaluation_df.apply(
    lambda row: judge_relevance(row["cv_text"], row["job_text"]),
    axis=1
)

In [69]:
evaluation_df.to_csv(
    "evaluation_with_labels.csv",
    index=False
)

In [70]:
print(
    evaluation_df["relevance"]
    .value_counts()
    .sort_index()
)

relevance
0     5
1     8
2     7
3    20
Name: count, dtype: int64


In [71]:
#Precision@10
evaluation_df["is_relevant"] = evaluation_df["relevance"].apply(lambda x: 1 if x >= 2 else 0)

precision_per_cv = evaluation_df.groupby("cv_name")["is_relevant"].mean()

print("Precision@10 per CV:\n", precision_per_cv)
print("\nAverage Precision@10:", precision_per_cv.mean())

Precision@10 per CV:
 cv_name
Cybersecurity        1.0
Data Analyst         0.8
Data Science         0.5
Software Engineer    0.4
Name: is_relevant, dtype: float64

Average Precision@10: 0.6749999999999999


In [72]:
# NDCG@10
from sklearn.metrics import ndcg_score
import numpy as np

ndcg_scores = []

for cv_name, group in evaluation_df.groupby("cv_name"):

    true_rel = group["relevance"].values.reshape(1, -1)
    pred_score = group["score"].values.reshape(1, -1)

    ndcg = ndcg_score(true_rel, pred_score, k=10)
    ndcg_scores.append(ndcg)

    print(cv_name, ndcg)

print("\nAverage NDCG@10:", np.mean(ndcg_scores))

Cybersecurity 0.9999999999999998
Data Analyst 0.9899338733403646
Data Science 0.8791816004949932
Software Engineer 0.7754362209453288

Average NDCG@10: 0.9111379236951715


In [73]:
!pip freeze > requirements.txt